<a href="https://colab.research.google.com/github/Fahad-Alam-Jamal/Flyrank_ML_Internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fahad-Alam-Jamal/Flyrank_ML_Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup (Colab or local)
On Colab this clones the repo and installs requirements. Locally it just moves to the repo root.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. Ranked actions + reason codes

*The queue is a human-review ranking, not an automated publishing decision. It orders pages by the strength of their refresh opportunity and the evidence the model or rules found.*

This playbook intentionally keeps the output grounded: the queue is a next-best-review list, and the reason codes tell the reviewer which evidence to inspect first.

The current queue is organized into three practical tiers:

- **Tier 1 — refresh now with focused review:** visible pages with measured decline and a second signal that makes them especially actionable, such as low CTR, low engagement, or model-derived decline risk.
- **Tier 2 — refresh candidate:** declining, visible pages that still need review but do not yet show the strongest CTR or engagement flags.
- **Tier 3 — monitor or review later:** pages that may be useful in a broader backlog, but should not be opened before Tier 1.

The reason codes in the queue are the human-facing evidence tags that explain why a row scored highly:
- `declining_with_demand`: declining while still attracting impressions. This is the core priority signal.
- `visible_model_opportunity`: the model sees a visible refresh opportunity.
- `model_decline_risk`: the model identifies decline risk beyond the simple trend bucket.
- `low_ctr_visible_page`: visible pages where clicks lag their position and impressions.
- `low_engagement_visible_page`: visible pages with weak engagement or scroll metrics.
- `ctr_review_candidate`: a review cue for titles, snippets, and search intent.
- `engagement_review_candidate`: a review cue for content quality, structure, or UX.
- `stale_visible_page`, `page_one_decay_risk`, `thin_visible_page`: supporting flags for age, page-one risk, and page thickness.

The queue is meant to surface the best review candidates, not to make a final editorial decision automatically.

In [2]:
import pandas as pd
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / '.git').exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

queue_path = repo_root / 'outputs' / 'refresh_queue.csv'
queue = pd.read_csv(queue_path)

print('Queue rows:', len(queue))
print('\nTop 5 ranked items:')
print(
    queue.loc[:4, ['final_rank', 'final_refresh_score', 'final_reason_codes', 'suggested_action', 'confidence']]
    .to_string(index=False)
)

reason_counts = queue['final_reason_codes'].str.split('|').explode().value_counts()
print('\nTop reason codes:')
print(reason_counts.head(10).to_string())

Queue rows: 30000

Top 5 ranked items:
 final_rank  final_refresh_score                                                                                                                                                   final_reason_codes       suggested_action confidence
          1            81.734212 declining_with_demand|low_ctr_visible_page|low_engagement_visible_page|model_decline_risk|visible_model_opportunity|ctr_review_candidate|engagement_review_candidate refresh_and_review_ctr       high
          2            81.603243                                                         declining_with_demand|low_ctr_visible_page|model_decline_risk|visible_model_opportunity|ctr_review_candidate refresh_and_review_ctr     medium
          3            81.544618                                                         declining_with_demand|low_ctr_visible_page|model_decline_risk|visible_model_opportunity|ctr_review_candidate refresh_and_review_ctr       high
          4            81.169731 

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Use this notebook output as a decision-support tool for editorial and content operations teams. The primary use case is:

- prioritize pages for refresh, rewrite, and editorial review;
- focus limited human attention on pages that still have visible search demand and measurable decline;
- surface the strongest evidence to the reviewer through reason codes and confidence labels.

This work is not valid for:

- automated content publishing, suppression, or deletion;
- interpreting the score as a guarantee that any one page will recover traffic;
- pages with little or no visibility in the data (low impressions, missing trend information, or otherwise outside the starter dataset coverage);
- a substitute for editorial judgment about campaign goals, brand fit, or legal content considerations.

The limits are real: the score is based on a historical snapshot and the starter label `trend_direction == "down"`. It is a recommendation list built from observed signals, not a causal experiment.

In [3]:
import pandas as pd
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / '.git').exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

queue_path = repo_root / 'outputs' / 'refresh_queue.csv'
queue = pd.read_csv(queue_path)

print('Confidence counts:')
print(queue['confidence'].value_counts().to_string())

print('\nSuggested action mix:')
print(queue['suggested_action'].value_counts().to_string())

Confidence counts:
confidence
low       15000
medium    11398
high       3602

Suggested action mix:
suggested_action
monitor                          13083
refresh                           8188
refresh_and_review_ctr            6654
refresh_and_review_engagement     1993
expand_and_refresh                  82


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Before acting on the queue, a human should verify at least these items:

- the page meaningfully fits the current editorial strategy and is not already being refreshed by a separate campaign;
- the page was not recently updated after the snapshot cutoff, since the dataset only reflects the prior 90-day window;
- the content format and intent are appropriate for the suggested review type (for example, a low CTR page may still be fine if it is an intentionally brand-led landing page);
- for `low_ctr_visible_page`, confirm whether the issue is query mismatch, SERP feature displacement, or a weak title/snippet;
- for `low_engagement_visible_page`, confirm whether the page has a long-form format that should be improved for readability, structure, or mobile experience.

The explicit no-go list:

- do not automate refresh, publish, remove, or rewrite actions based solely on the score;
- do not treat the queue as a final decision list without editorial triage;
- do not use this output to make technical SEO changes that require subject-matter review, such as rewriting legal or policy language;
- do not apply it to pages with insufficient traffic signal, low impressions, or missing trend data;
- do not use the score as a budget allocation rule or as an absolute measure of content quality.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

This is a light production readiness plan: the queue is for human review, and the monitoring signals are designed to tell the team when the recommendation list needs a fresh data refresh or a new validation run.

The simplest monitoring checks for this queue are:

- the top-ranked pages are repeatedly false positives in editorial review;
- the top 100 recommendations include a rising share of low-impression or low-visibility items;
- the distribution of reason codes shifts away from `declining_with_demand` or toward `general_refresh_review`;
- model validation metrics on a held-out split decrease significantly, especially precision@50 or average precision.

Retrain or rerun the queue when:

- a new 90-day snapshot becomes available;
- business priorities or editorial strategy change;
- the visible traffic mix shifts materially (for example, more page-one content or a different search intent mix);
- editorial review shows that the top-ranked pages are not the strongest refresh candidates.

In [5]:
import json
import shutil
from pathlib import Path

import pandas as pd

repo_root = Path.cwd()
while not (repo_root / '.git').exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

queue_path = repo_root / 'outputs' / 'refresh_queue.csv'
paper_output_dir = repo_root / 'work' / 'outputs'
figures_dir = repo_root / 'work' / 'figures'
chart_dir = repo_root / 'outputs' / 'charts'

paper_output_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

queue = pd.read_csv(queue_path)
queue.to_csv(paper_output_dir / 'refresh_queue_for_paper.csv', index=False)
queue.head(50).to_csv(paper_output_dir / 'refresh_queue_top50.csv', index=False)

metrics = {
    'rows_scored': len(queue),
    'top_score': float(queue['final_refresh_score'].max()),
    'median_score': float(queue['final_refresh_score'].median()),
    'confidence_counts': queue['confidence'].value_counts().to_dict(),
    'action_mix': queue['suggested_action'].value_counts().to_dict(),
    'top_reason_codes': queue['final_reason_codes'].str.split('|').explode().value_counts().head(10).to_dict(),
}

with open(paper_output_dir / 'refresh_queue_export_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

for chart_name in ['top_reason_codes.svg', 'action_mix.svg', 'confidence_mix.svg', 'top_feature_importance.svg']:
    src = chart_dir / chart_name
    if src.exists():
        shutil.copy(src, figures_dir / chart_name)

print('Wrote queue exports to', paper_output_dir)
print('Wrote chart copies to', figures_dir)
print('Metrics exported to', paper_output_dir / 'refresh_queue_export_metrics.json')

Wrote queue exports to /content/flyrank-ml-internship-starter/work/outputs
Wrote chart copies to /content/flyrank-ml-internship-starter/work/figures
Metrics exported to /content/flyrank-ml-internship-starter/work/outputs/refresh_queue_export_metrics.json


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

The exported queue is the exact ranked list the editorial team should use for review. The metrics file records the queue shape and the top reason codes. The copied figures are the visual evidence the paper can reuse without depending on the committed `outputs/` folder.

This notebook writes the ranked review queue and a small set of reusable assets into `work/outputs/` and `work/figures/` so the paper can trace the recommendation list back to concrete artifacts.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print('Run the export cell above to generate work/outputs/refresh_queue_for_paper.csv and work/figures/.')

Run the export cell above to generate work/outputs/refresh_queue_for_paper.csv and work/figures/.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.